<a href="https://colab.research.google.com/github/JGradone/Topics-Sentinel/blob/main/Redwing_ERDDAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Topics in Marine Science: Diving into Redwing's Glider Data
*Joe Gradone, Rutgers University*

**Redwing** is a Teledyne Webb Research Slocum glider flying the Sentinel mission, a Rutgers / Teledyne Webb attempt to be the first autonomous underwater vehicle to circumnavigate the globe. In this notebook you will download Redwing's real-time data straight from the Rutgers ERDDAP server and explore two sides of it:

1. **Engineering data**: how the glider flies (buoyancy pump, battery position, pitch), how healthy it is (battery, internal vacuum), and what currents it fought.
2. **Science data**: what the CTD (Conductivity–Temperature–Depth sensor) measured about the ocean: temperature, salinity, and density.

### Learning goals
By the end you should be able to:
* Load any glider dataset from ERDDAP into a `pandas` DataFrame
* Explain how a Slocum glider uses buoyancy and battery position to fly
* Separate a glider record into individual dive and climb profiles
* Make depth–time sections, profiles, and T–S diagrams

### How to use this notebook
Notebooks have **Markdown** cells (text, like this one) and **Code** cells. Run a code cell by clicking the ▶ button, pressing **Shift + Enter** (run and move to the next cell), or **Cmd/Ctrl + Enter** (run in place).

**Run the cells in order from the top.** Later cells use variables made in earlier ones. If something breaks, use *Runtime → Restart session and run all*.

Cells marked **✏️ Your turn** are for you. Change the values marked `# <- change me` and see what happens.

---
## 1. Setup

Google Colab already has `numpy`, `pandas`, and `matplotlib`. We need three extra packages:
* [`erddapy`](https://ioos.github.io/erddapy/): builds ERDDAP data requests for us
* [`gsw`](https://teos-10.github.io/GSW-Python/): the TEOS-10 seawater equation of state (salinity, density, ...)
* [`cmocean`](https://matplotlib.org/cmocean/): colormaps designed for oceanography

In [ ]:
%pip install --quiet erddapy gsw cmocean

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import cmocean.cm as cmo
import gsw
import io
import requests
from erddapy import ERDDAP

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

---
## 2. Where the data live: ERDDAP

[ERDDAP](https://slocum-data.marine.rutgers.edu/erddap/) is a data server used across oceanography. You ask it for a dataset, a list of variables, and (optionally) a time range, and it sends back a table.

Every glider deployment has a **dataset ID**. Redwing's real-time dataset is `redwing-20251011T1511-trajectory-raw-rt` (glider name, deployment date and time, then "raw real-time"). You can browse it in your web browser here:
<https://slocum-data.marine.rutgers.edu/erddap/tabledap/redwing-20251011T1511-trajectory-raw-rt.html>

The function below wraps up the request so you can reuse it. It works for **any** glider on this server: just change `dataset_id`.

In [ ]:
SERVER = 'https://slocum-data.marine.rutgers.edu/erddap'
DATASET_ID = 'redwing-20251011T1511-trajectory-raw-rt'


def read_csv_from_url(url, **kwargs):
    """pd.read_csv for a web address, with a fallback for servers whose SSL certificate
    Python can't verify. That's fine for public, read-only data like this."""
    try:
        return pd.read_csv(url, **kwargs)
    except Exception as err:
        if 'CERTIFICATE_VERIFY_FAILED' not in str(err):
            raise
        requests.packages.urllib3.disable_warnings()
        response = requests.get(url, verify=False, timeout=300)
        response.raise_for_status()
        return pd.read_csv(io.StringIO(response.text), **kwargs)


def load_glider_data(variables, start=None, end=None, dataset_id=DATASET_ID, server=SERVER):
    """Download glider variables from ERDDAP and return a pandas DataFrame.

    variables : list of variable names, e.g. ['time', 'm_depth']
    start, end : optional time limits as strings, e.g. '2025-10-20'
    """
    if 'time' not in variables:
        variables = ['time'] + list(variables)

    e = ERDDAP(server=server, protocol='tabledap', response='csv')
    e.dataset_id = dataset_id
    e.variables = variables
    if start is not None:
        e.constraints['time>='] = start
    if end is not None:
        e.constraints['time<='] = end

    url = e.get_download_url()
    df = read_csv_from_url(url, skiprows=[1])  # ERDDAP puts units in the 2nd row; skip it
    df['time'] = pd.to_datetime(df['time']).dt.tz_localize(None)  # all times are UTC
    return df.sort_values('time').reset_index(drop=True)


def list_glider_variables(dataset_id=DATASET_ID, server=SERVER):
    """Return a table of every variable in the dataset with its units and valid range."""
    e = ERDDAP(server=server, protocol='tabledap')
    info = read_csv_from_url(e.get_info_url(dataset_id, response='csv'))
    attrs = info[info['Row Type'] == 'attribute']
    table = attrs.pivot_table(index='Variable Name', columns='Attribute Name',
                              values='Value', aggfunc='first')
    names = info.loc[info['Row Type'] == 'variable', 'Variable Name']
    return table.reindex(names)[['units', 'actual_range']]

### What variables are there?

Slocum gliders log **hundreds** of variables. Their names follow a pattern:

| Prefix | Meaning | Example |
|---|---|---|
| `m_` | **measured** by the glider's flight computer | `m_depth`, `m_pitch`, `m_battery` |
| `c_` | **commanded**: what the glider was *told* to do | `c_de_oil_vol`, `c_heading` |
| `sci_` | measured by the **science** computer (the sensors) | `sci_water_temp`, `sci_water_cond` |
| `x_` | derived / computed onboard | `x_de_oil_flux` |

Some of the most useful ones:

| Variable | What it is | Units |
|---|---|---|
| `m_depth` | glider depth from its own pressure sensor | m |
| `m_de_oil_vol` | oil pumped into the external bladder by the **deep buoyancy engine**. Positive = more buoyant (go up), negative = less buoyant (go down) | cc |
| `m_battpos` | position of the battery pack that slides fore/aft to set **pitch** | inches |
| `m_pitch`, `m_roll`, `m_heading` | glider attitude | radians |
| `m_battery` | battery voltage | volts |
| `m_battery_amp_hours_remaining` | estimated remaining battery charge | amp-hours |
| `m_vacuum` | internal hull vacuum. A steady drop toward 0 can mean a **leak** | inHg |
| `m_veh_temp` | temperature inside the hull | °C |
| `m_water_vx`, `m_water_vy` | depth-averaged current (east, north) over each dive segment | m/s |
| `sci_water_temp` | CTD temperature | °C |
| `sci_water_cond` | CTD conductivity | S/m |
| `sci_water_pressure` | CTD pressure | bar |

Run the next cell to see the full list.

In [ ]:
variable_table = list_glider_variables()
print(f'{len(variable_table)} variables in {DATASET_ID}')
variable_table

---
## 3. Download Redwing's data

We'll grab a set of engineering and science variables for the whole mission. It's about 50 MB, so give it 10–30 seconds.

In [ ]:
engineering_vars = [
    'm_depth', 'm_pitch', 'm_roll', 'm_heading', 'm_battpos',
    'm_de_oil_vol', 'c_de_oil_vol',
    'm_battery', 'm_battery_amp_hours_remaining', 'm_vacuum', 'm_veh_temp',
    'm_water_vx', 'm_water_vy',
]
science_vars = ['sci_water_temp', 'sci_water_cond', 'sci_water_pressure']
position_vars = ['latitude', 'longitude']

raw = load_glider_data(['time'] + position_vars + engineering_vars + science_vars)
print(f'{len(raw):,} rows from {raw.time.min()} to {raw.time.max()} (UTC)')
raw.head(10)

### ⚠️ Why so many `NaN`s?

A Slocum glider has **two computers**. The *flight computer* records the `m_` and `c_` variables; the *science computer* records the `sci_` variables. They log at different moments, so each row usually has values from only one of them, and the rest are `NaN` ("not a number", i.e. missing).

Also, this is **real-time** data: to save satellite bandwidth, the glider only sends a subset of its measurements home each time it surfaces. Some variables (like `m_water_vx`) are only computed once per dive segment.

The easiest way to work with this is to split it into two tidy tables: one for **engineering** and one for **science**.

In [ ]:
raw[engineering_vars + science_vars].notna().sum().rename('number of valid values').to_frame()

---
## 4. Clean up and convert units

**Engineering:** angles come in radians, so we convert to degrees.

**Science:** we
* drop junk values (e.g. `0` temperature recorded before the CTD warmed up),
* convert pressure from bar to decibar (1 bar = 10 dbar; 1 dbar ≈ 1 m of depth),
* use `gsw` to compute **practical salinity**, **conservative temperature**, and **potential density anomaly** σ₀ (density − 1000 kg/m³, referenced to the surface).

In [ ]:
# ----- Engineering table -----
eng = raw[['time'] + position_vars + engineering_vars].dropna(subset=engineering_vars, how='all').copy()
for angle in ['m_pitch', 'm_roll', 'm_heading']:
    eng[angle] = np.degrees(eng[angle])

# ----- Science table -----
sci = raw[['time'] + position_vars + science_vars].dropna(subset=science_vars).copy()
sci = sci[(sci.sci_water_temp > 0) & (sci.sci_water_cond > 2)]  # remove obvious junk

sci['pressure'] = sci['sci_water_pressure'] * 10              # bar -> dbar
sci['depth'] = -gsw.z_from_p(sci['pressure'], sci['latitude'])  # m, positive down
sci['salinity'] = gsw.SP_from_C(sci['sci_water_cond'] * 10,    # S/m -> mS/cm
                                sci['sci_water_temp'], sci['pressure'])
sci = sci[(sci.salinity > 30) & (sci.salinity < 38)]          # remove conductivity spikes

SA = gsw.SA_from_SP(sci['salinity'], sci['pressure'], sci['longitude'], sci['latitude'])
sci['cons_temp'] = gsw.CT_from_t(SA, sci['sci_water_temp'], sci['pressure'])
sci['sigma0'] = gsw.sigma0(SA, sci['cons_temp'])

sci = sci.rename(columns={'sci_water_temp': 'temperature'}).reset_index(drop=True)

print(f'Engineering rows: {len(eng):,}')
print(f'Science rows:     {len(sci):,}')
sci.head()

---
## 5. Mission overview

Let's get the big picture first: where did Redwing go, and how deep did it dive?

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
sc = ax.scatter(raw.longitude, raw.latitude, c=mdates.date2num(raw.time), s=2, cmap='viridis')
ax.plot(raw.longitude.iloc[0], raw.latitude.iloc[0], 'g*', ms=16, mec='k', label='start')
ax.plot(raw.longitude.iloc[-1], raw.latitude.iloc[-1], 'r*', ms=16, mec='k', label='latest')
cb = fig.colorbar(sc, ax=ax, label='Date')
cb.ax.yaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
ax.set_aspect(1 / np.cos(np.radians(raw.latitude.mean())))  # so degrees of lon & lat look the same length
ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
ax.set_title('Redwing track')
ax.legend()
plt.show()

In [ ]:
# Distance travelled: add up the great-circle distance between consecutive positions
track = raw[['time', 'latitude', 'longitude']].dropna().drop_duplicates(['latitude', 'longitude'])
distance_km = np.nansum(gsw.distance(track.longitude.values, track.latitude.values)) / 1000
days = (raw.time.max() - raw.time.min()).total_seconds() / 86400

print(f'Mission record: {days:.0f} days, about {distance_km:,.0f} km along track '
      f'({distance_km / days:.0f} km/day)')
print(f'Deepest glider depth: {eng.m_depth.max():.0f} m')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(eng.time, eng.m_depth, '.', ms=1)
ax.invert_yaxis()
ax.set_ylabel('Depth (m)')
ax.set_title('Glider depth over the whole mission')
plt.show()

Each thin vertical streak is one **dive and climb** (the glider flies a sawtooth, or "yo", pattern down to ~900 m and back).

🤔 **Discuss:** Look at what happens to the data after mid-January. How does it change? What might the glider be doing, and why would the data look this way? (Hint: plot `eng.m_depth` and `sci.pressure` for February–May only.)

---
## 6. Engineering deep dive: how does a glider fly?

A glider has **no propeller**. It moves by changing its buoyancy:
1. At the surface, the **buoyancy engine** pulls oil *into* the hull (`m_de_oil_vol` goes negative). The glider is now denser than seawater and **sinks**.
2. The **battery pack** slides forward (`m_battpos`) to pitch the nose down (`m_pitch` negative).
3. The wings turn some of that sinking into forward motion, like a paper airplane.
4. At the bottom of the dive, the pump pushes oil *out* into an external bladder (`m_de_oil_vol` positive), the battery slides back, the nose pitches up, and the glider **climbs**.

Let's watch one full dive and climb.

In [ ]:
# ✏️ Pick a time window. Try a few different ones!
start = '2025-10-20 00:00'  # <- change me
end   = '2025-10-20 08:00'  # <- change me

seg = eng[(eng.time >= start) & (eng.time <= end)]

fig, axs = plt.subplots(4, 1, figsize=(11, 10), sharex=True)

d = seg.dropna(subset=['m_depth'])
axs[0].plot(d.time, d.m_depth, '.-', ms=3)
axs[0].invert_yaxis()
axs[0].set_ylabel('Depth (m)')

d = seg.dropna(subset=['m_de_oil_vol'])
axs[1].plot(d.time, d.m_de_oil_vol, '.-', ms=3, label='measured (m_de_oil_vol)')
d = seg.dropna(subset=['c_de_oil_vol'])
axs[1].plot(d.time, d.c_de_oil_vol, '--', label='commanded (c_de_oil_vol)')
axs[1].axhline(0, color='k', lw=0.8)
axs[1].set_ylabel('Oil volume (cc)')
axs[1].legend(loc='upper right')

d = seg.dropna(subset=['m_battpos'])
axs[2].plot(d.time, d.m_battpos, '.-', ms=3, color='C2')
axs[2].set_ylabel('Battery position (in)')

d = seg.dropna(subset=['m_pitch'])
axs[3].plot(d.time, d.m_pitch, '.-', ms=3, color='C3')
axs[3].axhline(0, color='k', lw=0.8)
axs[3].set_ylabel('Pitch (°)')

axs[-1].xaxis.set_major_formatter(mdates.DateFormatter('%m-%d\n%H:%M'))
fig.suptitle(f'Redwing flight: {start} to {end} UTC')
fig.tight_layout()
plt.show()

🤔 **Discuss:**
* What happens to the oil volume right before the glider turns around at the bottom?
* How does battery position relate to pitch?
* Is the glider dive angle the same going down as coming up?

### Vertical speed

We can compute how fast the glider moves up and down from the change in depth over time.

In [ ]:
d = eng.dropna(subset=['m_depth']).copy()
d['w'] = -d.m_depth.diff() / d.time.diff().dt.total_seconds()  # m/s, positive = up
d = d[(d.m_depth > 50) & d.w.between(-1, 1)]                     # ignore near-surface & glitches

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(d.w, d.m_depth, '.', ms=1, alpha=0.3)
ax.invert_yaxis()
ax.axvline(0, color='k', lw=0.8)
ax.set_xlabel('Vertical speed (m/s, + up)')
ax.set_ylabel('Depth (m)')
ax.set_title('Glider vertical speed, whole mission')
plt.show()

print(f"Typical dive speed:  {-d.loc[d.w < 0, 'w'].median():.3f} m/s downward")
print(f"Typical climb speed: {d.loc[d.w > 0, 'w'].median():.3f} m/s upward")

### Glider health: battery and vacuum

The battery decides how long the mission can last. The internal **vacuum** tells us if the hull is sealed: a steady drift toward 0 inHg can mean seawater is leaking in. Vacuum also changes with the temperature inside the hull (warm air expands), so we plot that too.

In [ ]:
fig, axs = plt.subplots(4, 1, figsize=(12, 11), sharex=True)

d = eng.dropna(subset=['m_battery_amp_hours_remaining'])
axs[0].plot(d.time, d.m_battery_amp_hours_remaining, '.', ms=2)
axs[0].set_ylabel('Charge remaining\n(amp-hours)')

d = eng.dropna(subset=['m_battery'])
axs[1].plot(d.time, d.m_battery, '.', ms=2, color='C1')
axs[1].set_ylabel('Battery (V)')

d = eng.dropna(subset=['m_vacuum'])
axs[2].plot(d.time, d.m_vacuum, '.', ms=2, color='C2')
axs[2].set_ylabel('Vacuum (inHg)')

d = eng.dropna(subset=['m_veh_temp'])
axs[3].plot(d.time, d.m_veh_temp, '.', ms=2, color='C3')
axs[3].set_ylabel('Hull temp (°C)')

fig.suptitle('Redwing health over the mission')
fig.tight_layout()
plt.show()

### ✏️ Your turn (Engineering group)
1. **Battery budget:** Fit a straight line to amp-hours remaining vs. time (hint: `np.polyfit`) for the diving part of the mission. How many amp-hours per day does Redwing use? At that rate, when would the battery run out?
2. Does the glider use energy faster when diving or when sitting at the surface? Compare the slope before and after mid-January.
3. Plot `m_vacuum` against `m_veh_temp`. Is the vacuum change explained by temperature, or is something else going on?
4. Does vertical speed change over the mission? Compare the first and last weeks of diving.

In [ ]:
# Example to get you started: amp-hours used per day, using all the data
d = eng.dropna(subset=['m_battery_amp_hours_remaining'])
days_since_start = (d.time - d.time.min()).dt.total_seconds() / 86400
slope, intercept = np.polyfit(days_since_start, d.m_battery_amp_hours_remaining, 1)
print(f'Average use: {-slope:.2f} amp-hours per day')

# Now try it for only part of the mission...

### Currents: what was the glider fighting?

Each time it surfaces, the glider compares where it *expected* to be (from dead reckoning) with where its GPS says it *is*. The difference is the **depth-averaged current** (`m_water_vx` east, `m_water_vy` north). Slocum gliders only swim at about 0.3–0.4 m/s horizontally, so strong currents matter a lot!

In [ ]:
cur = eng.dropna(subset=['m_water_vx', 'm_water_vy', 'latitude', 'longitude'])
cur = cur[(cur.m_water_vx != 0) | (cur.m_water_vy != 0)]
speed = np.hypot(cur.m_water_vx, cur.m_water_vy)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(raw.longitude, raw.latitude, color='0.7', lw=1, zorder=0)
q = ax.quiver(cur.longitude, cur.latitude, cur.m_water_vx, cur.m_water_vy, speed,
              cmap=cmo.speed, scale=25, width=0.0025)
ax.quiverkey(q, 0.1, 0.92, 0.5, '0.5 m/s', coordinates='axes')
fig.colorbar(q, ax=ax, label='Current speed (m/s)')
ax.set_aspect(1 / np.cos(np.radians(raw.latitude.mean())))
ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
ax.set_title('Depth-averaged currents along the track')
plt.show()

print(f'Median current speed: {speed.median():.2f} m/s, max: {speed.max():.2f} m/s')

🤔 **Discuss:** Where were the strongest currents? Which major current system is this? (Hint: zoom in with `ax.set_xlim(...)` and `ax.set_ylim(...)`.)

---
## 7. Science deep dive: what did the CTD see?

### Splitting the record into profiles

The CTD records continuously as the glider goes up and down. To make profiles, we cut the record wherever the glider changes direction. Each piece that spans more than 50 m is one **profile**, labelled as a **dive** (going down) or a **climb** (going up).

In [ ]:
def find_profiles(df, min_range=50, max_gap='15min'):
    """Label each row with a profile number and direction ('dive' or 'climb').

    A new profile starts whenever pressure switches from increasing to
    decreasing (or back), or when there is a time gap longer than max_gap.
    Pieces spanning less than min_range dbar are dropped (profile_id = -1).
    """
    df = df.sort_values('time').reset_index(drop=True)
    p = df['pressure'].rolling(5, center=True, min_periods=1).median()  # smooth out noise
    direction = np.sign(p.diff()).replace(0, np.nan).ffill().bfill()
    new_piece = (direction != direction.shift()) | (df['time'].diff() > pd.Timedelta(max_gap))
    piece = new_piece.cumsum()

    span = df.groupby(piece)['pressure'].transform(lambda x: x.max() - x.min())
    keep = span >= min_range
    df['profile_id'] = -1
    df.loc[keep, 'profile_id'] = pd.factorize(piece[keep])[0]
    df['direction'] = np.where(direction > 0, 'dive', 'climb')
    return df


sci = find_profiles(sci)
profiles = (sci[sci.profile_id >= 0]
            .groupby('profile_id')
            .agg(start=('time', 'min'), end=('time', 'max'), direction=('direction', 'first'),
                 max_depth=('depth', 'max'), n_points=('time', 'size'),
                 latitude=('latitude', 'mean'), longitude=('longitude', 'mean')))
print(f'Found {len(profiles)} profiles')
profiles.head(10)

🤔 **Discuss:** Run `profiles.direction.value_counts()`. Why are there so many more climbs than dives? (Hint: running the CTD costs battery power, so glider pilots often switch it on only for climbs. Plot `profiles.start` against `profiles.max_depth`, coloured by direction, to see when this changed.)

### Depth–time sections

A **section** plot shows the whole water column changing through time (or along the track). Colour = the measured property.

In [ ]:
# ✏️ Pick a time window for the section
start = '2025-10-11'  # <- change me
end   = '2026-01-17'  # <- change me

s = sci[(sci.time >= start) & (sci.time <= end)]

fig, axs = plt.subplots(3, 1, figsize=(12, 11), sharex=True)
for ax, var, cmap, label in zip(
        axs,
        ['temperature', 'salinity', 'sigma0'],
        [cmo.thermal, cmo.haline, cmo.dense],
        ['Temperature (°C)', 'Practical salinity', 'σ₀ (kg/m³)']):
    vmin, vmax = s[var].quantile([0.01, 0.99])
    sc = ax.scatter(s.time, s.depth, c=s[var], s=2, cmap=cmap, vmin=vmin, vmax=vmax)
    fig.colorbar(sc, ax=ax, label=label)
    ax.invert_yaxis()
    ax.set_ylabel('Depth (m)')
fig.suptitle(f'Redwing CTD sections, {start} to {end}')
fig.tight_layout()
plt.show()

### T–S diagram

Plotting temperature against salinity lets us identify **water masses**: bodies of water with a characteristic temperature and salinity from where they formed. The grey lines are constant density (σ₀).

In [ ]:
s = sci[(sci.time >= start) & (sci.time <= end)]

# density contours for the background
t_grid, s_grid = np.meshgrid(np.linspace(s.cons_temp.min() - 1, s.cons_temp.max() + 1, 100),
                             np.linspace(s.salinity.min() - 0.2, s.salinity.max() + 0.2, 100))
sig_grid = gsw.sigma0(s_grid, t_grid)  # using SP as an approximation of SA for the contours

fig, ax = plt.subplots(figsize=(8, 7))
cs = ax.contour(s_grid, t_grid, sig_grid, colors='0.6', linewidths=0.8, levels=np.arange(20, 30, 0.5))
ax.clabel(cs, fmt='%.1f', fontsize=8)
sc = ax.scatter(s.salinity, s.cons_temp, c=s.depth, s=2, cmap=cmo.deep)
fig.colorbar(sc, ax=ax, label='Depth (m)')
ax.set_xlabel('Practical salinity')
ax.set_ylabel('Conservative temperature (°C)')
ax.set_title('T–S diagram')
plt.show()

### A single dive and climb

Now we zoom in on one dive and the climb right after it. If the sensor were perfect, both should give the same profile (the glider is in nearly the same place). Differences between them can come from real ocean variability, or from **sensor thermal lag**: the conductivity cell takes time to warm/cool to the water temperature, which makes the computed salinity wrong where temperature changes quickly (e.g. in the thermocline).

In [ ]:
def plot_dive_climb(dive_id):
    """Plot temperature and salinity for a dive and the climb that follows it."""
    dive = sci[sci.profile_id == dive_id]
    climb = sci[sci.profile_id == dive_id + 1]
    if dive.empty or climb.empty:
        print(f'Profile {dive_id} or {dive_id + 1} does not exist. Check the `profiles` table.')
        return

    fig, axs = plt.subplots(1, 3, figsize=(13, 7), sharey=True)
    for ax, var, label in zip(axs, ['temperature', 'salinity', 'sigma0'],
                              ['Temperature (°C)', 'Practical salinity', 'σ₀ (kg/m³)']):
        ax.plot(dive[var], dive.depth, '.-', ms=3, label=f'{dive.direction.iloc[0]} (profile {dive_id})')
        ax.plot(climb[var], climb.depth, '.-', ms=3, label=f'{climb.direction.iloc[0]} (profile {dive_id + 1})')
        ax.set_xlabel(label)
    axs[0].invert_yaxis()
    axs[0].set_ylabel('Depth (m)')
    axs[0].legend(loc='lower right')
    fig.suptitle(f'Profiles {dive_id} and {dive_id + 1}: {dive.time.min():%Y-%m-%d %H:%M} UTC')
    fig.tight_layout()
    plt.show()


# We need full-depth dives (the first days were shallow test dives) that are followed
# right away by a climb (often the CTD was only on for climbs)
next_is_climb = (profiles.direction.shift(-1) == 'climb') & \
                (profiles.start.shift(-1) - profiles.end < pd.Timedelta('10min'))
deep_dives = profiles[(profiles.direction == 'dive') & (profiles.max_depth > 800) & next_is_climb]
print(f'{len(deep_dives)} full-depth dive + climb pairs. The first few:')
display(deep_dives.head())

# ✏️ Pick a dive: deep_dives.index[0] is the first full-depth dive, [1] the second, ...
plot_dive_climb(deep_dives.index[10])  # <- change me

### ✏️ Your turn (Dive/Climb group)
1. Look at several dive–climb pairs. In which depth range do the dive and climb salinity disagree the most? Is that where temperature changes fastest?
2. Quantify it: interpolate the dive and climb onto the same depth grid (hint: `np.interp`, depth must be increasing) and plot the **difference** in salinity vs. depth. Average the difference over many pairs.
3. Is the difference always the same sign? What would you expect from thermal lag on a dive vs. a climb through a thermocline where temperature *decreases* with depth?
4. Plot a single dive and climb on a T–S diagram. Thermal lag often shows up as two separated "loops".

---
## 8. Going further
* **Other gliders:** browse all datasets at <https://slocum-data.marine.rutgers.edu/erddap/tabledap/index.html> and pass a different `dataset_id` to `load_glider_data(...)`.
* **More variables:** `variable_table` lists everything Redwing records. Try `m_roll`, `m_heading`, `x_de_oil_flux`, or `m_leakdetect_voltage`.
* **Save your work:** `sci.to_csv('redwing_science.csv')` writes a file you can download from the Colab file browser (folder icon on the left). In Colab, use *File → Save a copy in Drive* to keep your edits.